In [103]:
# Direct wrappers
#
# https://tslearn.readthedocs.io/en/latest/gen_modules/tslearn.clustering.html#module-tslearn.clustering
# https://www.sktime.net/en/latest/api_reference/auto_generated/sktime.clustering.k_medoids.TimeSeriesKMedoids.html#sktime.clustering.k_medoids.TimeSeriesKMedoids


from tslearn.clustering import TimeSeriesKMeans
from tslearn.clustering import KShape
from tslearn.clustering import KernelKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance, TimeSeriesResampler
from tslearn.datasets import CachedDatasets
from tslearn.utils import to_time_series_dataset

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
AKI_PATH = os.environ['AKI_PATH']
os.chdir(AKI_PATH)

ts_data = pd.read_parquet("ts.parquet.snappy")
ts_data = ts_data.assign(ID=ts_data.ID.astype('int64'))

In [ ]:
# X = [n_ts, sz, dim]
# if there is no interpolation we assume that the time dimension is the second one
# i.e. we have at least 2 dimensions

ts_ = ts_data.groupby('ID').apply(lambda x: x.drop(columns='ID').values)

In [ ]:
# Make X = [n_ts, sz, dim]
X = np.array([np.array(ts_.iloc[i]) for i in range(len(ts_))])
Xts = to_time_series_dataset(X)

In [ ]:
Normaliser = TimeSeriesScalerMeanVariance(mu=0., std=1.)  # Rescale time series
Xts = Normaliser.fit_transform(Xts)

In [ ]:
seed =0
# DBA-k-means
ts_km = TimeSeriesKMeans(n_clusters=3,
                          n_init=2,
                          metric="dtw", # "softdtw"
                          verbose=True,
                          n_jobs= 4,
                          max_iter_barycenter=10,
                          random_state=seed) # can handle mismatching lengths

ts_ks = KShape(n_clusters=3, verbose=True, random_state=seed) # cannot handle mismatching lengths

ts_gak = KernelKMeans(n_clusters=3, 
                      kernel="gak",# "rbf", "laplacian", "linear", "cosine"
                      random_state=seed)


In [ ]:
ts_km.fit(Xts)

In [96]:
LabelDF = pd.DataFrame(ts_km.labels_, columns=['Label'], index=ts_.index)
ts_data_merged = ts_data.merge(LabelDF, left_on='ID', right_index=True)

In [101]:
ts_data_merged